<a href="https://colab.research.google.com/github/Anupamjoshi14/Anu/blob/main/Copy_of_Untitled155.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip uninstall -y numpy pandas gradio bcrypt
!pip install numpy==1.26.4 pandas==2.2.2 gradio==4.29.0 bcrypt

import gradio as gr
import sqlite3
import datetime
import pandas as pd
import bcrypt
import re

# Initialize SQLite database
def init_db():
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    # Users table
    c.execute('''CREATE TABLE IF NOT EXISTS users (
                 id INTEGER PRIMARY KEY AUTOINCREMENT,
                 username TEXT UNIQUE NOT NULL,
                 password TEXT NOT NULL,
                 email TEXT,
                 avatar_url TEXT DEFAULT 'https://via.placeholder.com/50')''')
    # Groups table
    c.execute('''CREATE TABLE IF NOT EXISTS groups (
                 id INTEGER PRIMARY KEY AUTOINCREMENT,
                 name TEXT UNIQUE NOT NULL,
                 description TEXT)''')
    # Group memberships table
    c.execute('''CREATE TABLE IF NOT EXISTS group_members (
                 user_id INTEGER,
                 group_id INTEGER,
                 FOREIGN KEY (user_id) REFERENCES users (id),
                 FOREIGN KEY (group_id) REFERENCES groups (id),
                 PRIMARY KEY (user_id, group_id))''')
    # Posts table
    c.execute('''CREATE TABLE IF NOT EXISTS posts (
                 id INTEGER PRIMARY KEY AUTOINCREMENT,
                 user_id INTEGER,
                 group_id INTEGER,
                 content TEXT NOT NULL,
                 created_at TIMESTAMP,
                 FOREIGN KEY (user_id) REFERENCES users (id),
                 FOREIGN KEY (group_id) REFERENCES groups (id))''')
    # Tags table
    c.execute('''CREATE TABLE IF NOT EXISTS tags (
                 post_id INTEGER,
                 user_id INTEGER,
                 FOREIGN KEY (post_id) REFERENCES posts (id),
                 FOREIGN KEY (user_id) REFERENCES users (id),
                 PRIMARY KEY (post_id, user_id))''')
    # Likes table
    c.execute('''CREATE TABLE IF NOT EXISTS likes (
                 post_id INTEGER,
                 user_id INTEGER,
                 FOREIGN KEY (post_id) REFERENCES posts (id),
                 FOREIGN KEY (user_id) REFERENCES users (id),
                 PRIMARY KEY (post_id, user_id))''')
    # Reposts table
    c.execute('''CREATE TABLE IF NOT EXISTS reposts (
                 post_id INTEGER,
                 user_id INTEGER,
                 created_at TIMESTAMP,
                 FOREIGN KEY (post_id) REFERENCES posts (id),
                 FOREIGN KEY (user_id) REFERENCES users (id),
                 PRIMARY KEY (post_id, user_id))''')
    # Notifications table
    c.execute('''CREATE TABLE IF NOT EXISTS notifications (
                 id INTEGER PRIMARY KEY AUTOINCREMENT,
                 user_id INTEGER,
                 post_id INTEGER,
                 message TEXT,
                 created_at TIMESTAMP,
                 read BOOLEAN DEFAULT 0,
                 FOREIGN KEY (user_id) REFERENCES users (id),
                 FOREIGN KEY (post_id) REFERENCES posts (id))''')
    conn.commit()
    conn.close()

# User registration
def register_user(username, password, email):
    if not username or not password:
        return "❌ Username and password are required!", None
    hashed_password = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    try:
        c.execute("INSERT INTO users (username, password, email) VALUES (?, ?, ?)",
                  (username, hashed_password, email))
        conn.commit()
        return "✅ Registration successful! Please login.", None
    except sqlite3.IntegrityError:
        return "❌ Username already exists!", None
    finally:
        conn.close()

# User login
def login_user(username, password):
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    c.execute("SELECT id, password FROM users WHERE username = ?", (username,))
    user = c.fetchone()
    conn.close()
    if user and bcrypt.checkpw(password.encode('utf-8'), user[1].encode('utf-8')):
        return user[0], "✅ Login successful!"
    return None, "❌ Invalid credentials!"

# Create a group
def create_group(user_id, group_name, group_description):
    if not user_id:
        return "❌ Please login first!", None
    if not group_name:
        return "❌ Group name is required!", None
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    try:
        c.execute("INSERT INTO groups (name, description) VALUES (?, ?)",
                  (group_name, group_description))
        group_id = c.lastrowid
        c.execute("INSERT INTO group_members (user_id, group_id) VALUES (?, ?)",
                  (user_id, group_id))
        conn.commit()
        return f"✅ Group '{group_name}' created successfully!", None
    except sqlite3.IntegrityError:
        return "❌ Group name already exists!", None
    finally:
        conn.close()

# Join a group
def join_group(user_id, group_name):
    if not user_id:
        return "❌ Please login first!", None
    if not group_name:
        return "❌ Group name is required!", None
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    c.execute("SELECT id FROM groups WHERE name = ?", (group_name,))
    group = c.fetchone()
    if not group:
        conn.close()
        return "❌ Group not found!", None
    group_id = group[0]
    try:
        c.execute("INSERT INTO group_members (user_id, group_id) VALUES (?, ?)",
                  (user_id, group_id))
        conn.commit()
        return f"✅ Successfully joined '{group_name}'!", None
    except sqlite3.IntegrityError:
        return "❌ You are already a member of this group!", None
    finally:
        conn.close()

# Create a post with tagging
def create_post(user_id, group_name, content):
    if not user_id:
        return "❌ Please login first!", None
    if not group_name or not content:
        return "❌ Group name and content are required!", None
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    c.execute("SELECT g.id FROM groups g JOIN group_members gm ON g.id = gm.group_id WHERE g.name = ? AND gm.user_id = ?",
              (group_name, user_id))
    group = c.fetchone()
    if not group:
        conn.close()
        return "❌ You are not a member of this group or group does not exist!", None
    group_id = group[0]
    created_at = datetime.datetime.now()
    c.execute("INSERT INTO posts (user_id, group_id, content, created_at) VALUES (?, ?, ?, ?)",
              (user_id, group_id, content, created_at))
    post_id = c.lastrowid

    # Process tags
    tagged_usernames = re.findall(r'@(\w+)', content)
    for username in tagged_usernames:
        c.execute("SELECT id FROM users WHERE username = ?", (username,))
        tagged_user = c.fetchone()
        if tagged_user:
            tagged_user_id = tagged_user[0]
            c.execute("INSERT OR IGNORE INTO tags (post_id, user_id) VALUES (?, ?)",
                      (post_id, tagged_user_id))
            # Create notification
            c.execute("INSERT INTO notifications (user_id, post_id, message, created_at) VALUES (?, ?, ?, ?)",
                      (tagged_user_id, post_id, f"You were tagged in a post by {user_id}", created_at))
    conn.commit()
    conn.close()
    return "✅ Post created successfully!", None

# Like a post
def like_post(user_id, post_id):
    if not user_id:
        return "❌ Please login first!"
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    try:
        c.execute("INSERT INTO likes (post_id, user_id) VALUES (?, ?)",
                  (post_id, user_id))
        conn.commit()
        return "✅ Post liked!"
    except sqlite3.IntegrityError:
        c.execute("DELETE FROM likes WHERE post_id = ? AND user_id = ?",
                  (post_id, user_id))
        conn.commit()
        return "✅ Like removed!"
    finally:
        conn.close()

# Repost a post
def repost_post(user_id, post_id):
    if not user_id:
        return "❌ Please login first!"
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    c.execute("SELECT group_id, content FROM posts WHERE id = ?", (post_id,))
    post = c.fetchone()
    if not post:
        conn.close()
        return "❌ Post not found!"
    group_id, content = post
    created_at = datetime.datetime.now()
    try:
        c.execute("INSERT INTO reposts (post_id, user_id, created_at) VALUES (?, ?, ?)",
                  (post_id, user_id, created_at))
        c.execute("INSERT INTO posts (user_id, group_id, content, created_at) VALUES (?, ?, ?, ?)",
                  (user_id, group_id, f"Repost: {content}", created_at))
        conn.commit()
        return "✅ Post reposted!"
    except sqlite3.IntegrityError:
        conn.commit()
        return "❌ Already reposted!"
    finally:
        conn.close()

# View timeline feed
def view_timeline(user_id):
    if not user_id:
        return "Please login first!"
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    df = pd.read_sql_query("""
        SELECT p.id, p.content, p.created_at, u.username, u.avatar_url, g.name as group_name,
               (SELECT COUNT(*) FROM likes l WHERE l.post_id = p.id) as like_count,
               (SELECT COUNT(*) FROM reposts r WHERE r.post_id = p.id) as repost_count
        FROM posts p
        JOIN users u ON p.user_id = u.id
        JOIN groups g ON p.group_id = g.id
        JOIN group_members gm ON gm.group_id = p.group_id
        WHERE gm.user_id = ?
        ORDER BY p.created_at DESC
    """, conn, params=(user_id,))
    conn.close()
    return df if not df.empty else "No posts in your groups yet."

# View notifications
def view_notifications(user_id):
    if not user_id:
        return "Please login first!"
    conn = sqlite3.connect('social_platform.db')
    c = conn.cursor()
    df = pd.read_sql_query("""
        SELECT n.message, n.created_at, p.content as post_content
        FROM notifications n
        JOIN posts p ON n.post_id = p.id
        WHERE n.user_id = ? AND n.read = 0
        ORDER BY n.created_at DESC
    """, conn, params=(user_id,))
    c.execute("UPDATE notifications SET read = 1 WHERE user_id = ?", (user_id,))
    conn.commit()
    conn.close()
    return df if not df.empty else "No new notifications."

# Custom CSS for X-like UI
custom_css = """
/* Base styles */
body, .gradio-container {
    font-family: 'Inter', sans-serif !important;
}

/* Light theme */
:root, .light-theme {
    --background: #ffffff;
    --card-bg: #ffffff;
    --text: #000000;
    --primary: #1DA1F2;
    --border: #e6ecf0;
    --shadow: rgba(0, 0, 0, 0.1);
}

/* Dark theme */
.dark-theme {
    --background: #15202b;
    --card-bg: #192734;
    --text: #ffffff;
    --primary: #1DA1F2;
    --border: #38444d;
    --shadow: rgba(0, 0, 0, 0.3);
}

.gradio-container {
    background: var(--background) !important;
    color: var(--text) !important;
    max-width: 800px !important;
    margin: 0 auto !important;
    padding: 20px !important;
}

.card {
    background: var(--card-bg) !important;
    border: 1px solid var(--border) !important;
    border-radius: 12px !important;
    padding: 15px !important;
    margin-bottom: 15px !important;
    display: flex;
    align-items: flex-start;
}

.post-container {
    display: flex;
    width: 100%;
}

.avatar {
    width: 50px;
    height: 50px;
    border-radius: 50%;
    margin-right: 15px;
}

.post-content {
    flex-grow: 1;
}

.post-header {
    display: flex;
    justify-content: space-between;
    margin-bottom: 5px;
}

.username {
    font-weight: bold;
    color: var(--text);
}

.timestamp {
    color: #657786;
    font-size: 0.9em;
}

.group-name {
    color: #657786;
    font-size: 0.9em;
    margin-left: 10px;
}

.post-text {
    margin: 10px 0;
    word-wrap: break-word;
}

.tag {
    color: var(--primary);
    font-weight: bold;
}

.post-actions {
    display: flex;
    justify-content: space-between;
    margin-top: 10px;
}

.action-button {
    background: none !important;
    color: #657786 !important;
    border: none !important;
    padding: 5px !important;
    cursor: pointer;
}

.action-button:hover {
    color: var(--primary) !important;
}

button {
    background: var(--primary) !important;
    color: white !important;
    border: none !important;
    border-radius: 20px !important;
    padding: 8px 16px !important;
    transition: background 0.2s !important;
}

button:hover {
    background: #1a91da !important;
}

input, textarea {
    background: var(--card-bg) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    color: var(--text) !important;
    padding: 10px !important;
}

.dataframe {
    background: var(--card-bg) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
}

#theme-toggle {
    position: fixed;
    top: 20px;
    right: 20px;
    background: var(--primary) !important;
    padding: 10px !important;
    border-radius: 50% !important;
    cursor: pointer;
}
"""

# Custom JavaScript for theme toggle and tag highlighting
custom_js = """
<script>
function toggleTheme() {
    const container = document.querySelector('.gradio-container');
    if (container.classList.contains('dark-theme')) {
        container.classList.remove('dark-theme');
        container.classList.add('light-theme');
        localStorage.setItem('theme', 'light');
    } else {
        container.classList.remove('light-theme');
        container.classList.add('dark-theme');
        localStorage.setItem('theme', 'dark');
    }
}

document.addEventListener('DOMContentLoaded', () => {
    const savedTheme = localStorage.getItem('theme') || 'light';
    const container = document.querySelector('.gradio-container');
    container.classList.add(savedTheme + '-theme');

    // Highlight tags in post content
    document.querySelectorAll('.post-text').forEach(el => {
        el.innerHTML = el.innerHTML.replace(/@(\w+)/g, '<span class="tag">@$1</span>');
    });
});
</script>
"""

# Gradio interface
def main_interface():
    init_db()
    user_id = gr.State(value=None)

    def handle_login(username, password):
        uid, message = login_user(username, password)
        return uid, message, gr.update(value="", type="password"), gr.update(value="")

    def handle_register(username, password, email):
        message, _ = register_user(username, password, email)
        return message, gr.update(value=""), gr.update(value="", type="password"), gr.update(value="")

    def handle_create_group(user_id, group_name, group_description):
        message, _ = create_group(user_id, group_name, group_description)
        return message, gr.update(value=""), gr.update(value="")

    def handle_join_group(user_id, group_name):
        message, _ = join_group(user_id, group_name)
        return message, gr.update(value="")

    def handle_create_post(user_id, group_name, content):
        message, _ = create_post(user_id, group_name, content)
        return message, gr.update(value=""), gr.update(value="")

    def handle_like_post(user_id, post_id):
        return like_post(user_id, post_id)

    def handle_repost_post(user_id, post_id):
        return repost_post(user_id, post_id)

    def render_timeline(df):
        if isinstance(df, str):
            return df
        html = ""
        for _, row in df.iterrows():
            html += f"""
            <div class='card'>
                <div class='post-container'>
                    <img src='{row['avatar_url']}' class='avatar'>
                    <div class='post-content'>
                        <div class='post-header'>
                            <span class='username'>{row['username']}</span>
                            <span class='group-name'>in {row['group_name']}</span>
                            <span class='timestamp'>{row['created_at']}</span>
                        </div>
                        <div class='post-text'>{row['content']}</div>
                        <div class='post-actions'>
                            <button class='action-button' onclick='triggerLike({row['id']})'>❤️ {row['like_count']}</button>
                            <button class='action-button' onclick='triggerRepost({row['id']})'>🔄 {row['repost_count']}</button>
                        </div>
                    </div>
                </div>
            </div>
            """
        return html

    with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as app:
        gr.HTML("<h1 style='text-align: center;'>Social Circle</h1>" + custom_js)
        gr.HTML('<button id="theme-toggle" onclick="toggleTheme()">🌙</button>')
        user_id_state = user_id

        # Tabs for navigation
        with gr.Tabs():
            with gr.TabItem("Home"):
                with gr.Group(elem_classes="card"):
                    gr.Markdown("## What's on your mind?")
                    post_group_name = gr.Textbox(label="Group Name", placeholder="Enter group name")
                    post_content = gr.Textbox(label="Post", placeholder="Share your thoughts... Use @username to tag", lines=3)
                    create_post_button = gr.Button("Post")
                    post_status = gr.Textbox(label="Status", interactive=False)

                gr.Markdown("## Timeline")
                timeline_button = gr.Button("Refresh Timeline")
                timeline_output = gr.HTML()

            with gr.TabItem("Notifications"):
                notifications_button = gr.Button("View Notifications")
                notifications_output = gr.Dataframe(label="Notifications", interactive=False)

            with gr.TabItem("Groups"):
                with gr.Group(elem_classes="card"):
                    gr.Markdown("## Manage Groups")
                    with gr.Row():
                        with gr.Column():
                            gr.Markdown("### Create Group")
                            group_name_create = gr.Textbox(label="Group Name", placeholder="Enter group name")
                            group_description = gr.Textbox(label="Description", placeholder="Describe your group")
                            create_group_button = gr.Button("Create Group")
                            create_group_status = gr.Textbox(label="Status", interactive=False)
                        with gr.Column():
                            gr.Markdown("### Join Group")
                            group_name_join = gr.Textbox(label="Group Name", placeholder="Enter group name to join")
                            join_group_button = gr.Button("Join Group")
                            join_group_status = gr.Textbox(label="Status", interactive=False)

            with gr.TabItem("Account"):
                with gr.Group(elem_classes="card"):
                    gr.Markdown("## Login / Register")
                    with gr.Row():
                        with gr.Column():
                            gr.Markdown("### Login")
                            login_username = gr.Textbox(label="Username", placeholder="Enter username")
                            login_password = gr.Textbox(label="Password", type="password", placeholder="Enter password")
                            login_button = gr.Button("Login")
                            login_status = gr.Textbox(label="Status", interactive=False)
                        with gr.Column():
                            gr.Markdown("### Register")
                            reg_username = gr.Textbox(label="Username", placeholder="Choose username")
                            reg_password = gr.Textbox(label="Password", type="password", placeholder="Choose password")
                            reg_email = gr.Textbox(label="Email (optional)", placeholder="Enter email")
                            reg_button = gr.Button("Register")
                            reg_status = gr.Textbox(label="Status", interactive=False)

        # Hidden components for like/repost triggers
        like_trigger = gr.Number(visible=False)
        repost_trigger = gr.Number(visible=False)

        # Event handlers
        login_button.click(
            fn=handle_login,
            inputs=[login_username, login_password],
            outputs=[user_id_state, login_status, login_password, login_username]
        )
        reg_button.click(
            fn=handle_register,
            inputs=[reg_username, reg_password, reg_email],
            outputs=[reg_status, reg_username, reg_password, reg_email]
        )
        create_group_button.click(
            fn=handle_create_group,
            inputs=[user_id_state, group_name_create, group_description],
            outputs=[create_group_status, group_name_create, group_description]
        )
        join_group_button.click(
            fn=handle_join_group,
            inputs=[user_id_state, group_name_join],
            outputs=[join_group_status, group_name_join]
        )
        create_post_button.click(
            fn=handle_create_post,
            inputs=[user_id_state, post_group_name, post_content],
            outputs=[post_status, post_group_name, post_content]
        )
        timeline_button.click(
            fn=lambda user_id: render_timeline(view_timeline(user_id)),
            inputs=[user_id_state],
            outputs=timeline_output
        )
        notifications_button.click(
            fn=view_notifications,
            inputs=[user_id_state],
            outputs=notifications_output
        )
        like_trigger.change(
            fn=handle_like_post,
            inputs=[user_id_state, like_trigger],
            outputs=gr.Textbox(visible=False)
        )
        repost_trigger.change(
            fn=handle_repost_post,
            inputs=[user_id_state, repost_trigger],
            outputs=gr.Textbox(visible=False)
        )

    return app

# Launch the app
if __name__ == "__main__":
    app = main_interface()
    app.launch(share=True)